In [1]:
import skripsi_forensik
import time
import pandas as pd

# Inisialisasi engine Forensik
engine = skripsi_forensik.ForensicEngine()
path_dataset = "/mnt/c/Documents and Settings/Ariekany/Downloads/Slicing Data BQ/"

print("=== [STEP 1] LOADING DATA & BENCHMARK PERFORMANCE ===")
start_time = time.time()

# Eksekusi Map-Reduce Rayon di Rust
total_tx = engine.load_data(path_dataset)

end_time = time.time()
durasi = end_time - start_time

print(f"Status: Berhasil memproses {total_tx:,} transaksi.")
print(f"Waktu Eksekusi: {durasi:.2f} detik")
print(f"Kecepatan: {int(total_tx/durasi):,} transaksi/detik")
print("-" * 50)

=== [STEP 1] LOADING DATA & BENCHMARK PERFORMANCE ===
Memulai ekstraksi paralel pada 5 file CSV...
Status: Berhasil memproses 4,771,819 transaksi.
Waktu Eksekusi: 31.04 detik
Kecepatan: 153,721 transaksi/detik
--------------------------------------------------
Fase Map selesai. 4771819 transaksi siap dirakit...
Fase Reduce selesai. Graf & Benford State siap di RAM!


In [2]:
print("=== [STEP 2] SYBIL CLUSTERING (UNION-FIND) ===")
# Mencari klaster dengan minimal 500 alamat (Sybil Threshold)
threshold = 500
hasil_klaster = engine.get_sybil_clusters(threshold)

# Mengurutkan berdasarkan ukuran klaster terbesar
sorted_clusters = sorted(hasil_klaster.items(), key=lambda x: x[1], reverse=True)

print(f"Ditemukan {len(sorted_clusters)} Entitas Sybil Raksasa (> {threshold} dompet).")
print("\nTop 5 Entitas Terduga Sindikat PlusToken:")
for i, (root, size) in enumerate(sorted_clusters[:10], 1):
    print(f"{i}. Root Address: {root} | Cluster Size: {size} dompet")

# Menyimpan target utama untuk pelacakan DFS nanti
target_sidik_jari = sorted_clusters[0][0]

=== [STEP 2] SYBIL CLUSTERING (UNION-FIND) ===
Ditemukan 393 Entitas Sybil Raksasa (> 500 dompet).

Top 5 Entitas Terduga Sindikat PlusToken:
1. Root Address: 3FFzQcacnbwsHn2rfVfPjHeDcyfNY4i5n2 | Cluster Size: 152939 dompet
2. Root Address: bc1qxymycl3zjpg36d9tarah70hm568tpjvp02w94t | Cluster Size: 121281 dompet
3. Root Address: 3L2icfh1d8YMcBjQVEfJkzSj2D8oaxViCA | Cluster Size: 104215 dompet
4. Root Address: 1LiYc5Sxf8wFbMD2s6MMWm4NmbeB1UDTv2 | Cluster Size: 94966 dompet
5. Root Address: 3MADuRNhQ3R8T96UxRY5TWwavJ5DmKbjd1 | Cluster Size: 89719 dompet
6. Root Address: 38LknLwosSAZSHaFYCdWc1sEm5PrgUtY79 | Cluster Size: 85156 dompet
7. Root Address: 3HcJs6pRaiK8boXjJHCfxkCNd1A7Us4MBE | Cluster Size: 69564 dompet
8. Root Address: 3CFjiMPXkK5jqEc8ZVfGHeepsN2sEUyr4v | Cluster Size: 40457 dompet
9. Root Address: bc1qcup0nlcjcamxfrzslhxu9s626gvmnxccgqucqf | Cluster Size: 35045 dompet
10. Root Address: 3NE7psWdpoM63ALsu7Tzb8BhfV5tAWUeTi | Cluster Size: 29755 dompet


In [3]:
print("=== [STEP 3] TRANSACTION TRACKING (DFS) - FOKUS HOT WALLET ===")

# Kita langsung tembak Peringkat 2 yang terbukti memecah dana secara agresif
target_hot_wallet = sorted_clusters[1][0] 
max_depth = 5
limit_exchange = 10000

print(f"Melacak rute pelarian dana agresif dari: {target_hot_wallet}...")
jalur_pelarian = engine.run_dfs(target_hot_wallet, max_depth, limit_exchange)

print(f"Pelacakan Selesai. Ditemukan {len(jalur_pelarian):,} rute pelarian unik!")

=== [STEP 3] TRANSACTION TRACKING (DFS) - FOKUS HOT WALLET ===
Melacak rute pelarian dana agresif dari: bc1qxymycl3zjpg36d9tarah70hm568tpjvp02w94t...
Pelacakan Selesai. Ditemukan 78,875 rute pelarian unik!


In [4]:
"""
real
Step 4 of 5: Interactive Network Visualization (Advanced Forensic Intelligence).

This step builds an interactive directed network graph using PyVis to visualize
the escape routes discovered in Step 3. The graph encodes forensic insights:
- Episcenter detection (primary target hot wallet).
- Cycle detection (round‑tripping / mixer behaviour) highlighted as triangles.
- Hop‑based colouring (dark red → steel blue) to indicate distance from the source.
- Edge thickness and tooltips show estimated transaction volume and BTC value.

This version uses a white background with dark text for improved print/readability.
The resulting HTML file is saved and automatically opened in the default browser.
"""

from pyvis.network import Network
import networkx as nx
from collections import Counter
import os
import random

print("=== [STEP 4] INTERACTIVE NETWORK VISUALIZATION (HEATMAP, VOLUME & VALUE) ===")

# ---------------------------------------------------------------------------
# 0. FORENSIC INTELLIGENCE: Automatic Episcenter & Cycle Detection
# ---------------------------------------------------------------------------
# The primary hot wallet is taken as the first node of the first escape route.
target_hot_wallet = jalur_pelarian[0][0] if len(jalur_pelarian) > 0 else "UNKNOWN"

# Build a NetworkX directed graph to find complex laundering cycles (round-tripping)
nx_graph = nx.DiGraph()
for route in jalur_pelarian[:200]:
    for i in range(len(route) - 1):
        nx_graph.add_edge(route[i], route[i+1])

# Detect all nodes involved in any directed cycle using NetworkX
cycle_nodes = set()
# nx.simple_cycles will find all elementary circuits in the directed graph
for cycle in nx.simple_cycles(nx_graph):
    for node in cycle:
        cycle_nodes.add(node)

# ---------------------------------------------------------------------------
# 1. Compute Node and Edge Frequencies (traffic heatmap)
# ---------------------------------------------------------------------------
node_frequencies = Counter()
edge_frequencies = Counter()

for route in jalur_pelarian[:200]:
    for i in range(len(route)):
        node_frequencies[route[i]] += 1

        if i < len(route) - 1:
            u, v = route[i], route[i+1]
            edge_frequencies[(u, v)] += 1

# ---------------------------------------------------------------------------
# Initialise the PyVis canvas (white background, dark text)
# ---------------------------------------------------------------------------
net = Network(
    height="100vh",
    width="100%",
    bgcolor="#ffffff",
    font_color="#333333",
    directed=True,
    notebook=False,
    cdn_resources="remote"
)

added_nodes = set()
added_edges = set()

# ---------------------------------------------------------------------------
# 2. Dynamic Colour Palette by Hop Distance (white‑background friendly)
# ---------------------------------------------------------------------------
def get_hop_color(hop):
    """
    Return a colour string based on hop distance from the source.
    Darker reds indicate proximity to the hot wallet; blue indicates safer distance.
    """
    if hop <= 1: return "#B30000"    # Hop 0/1: Dark red
    elif hop == 2: return "#E34A33"  # Hop 2: Brick red
    elif hop == 3: return "#FC8D59"  # Hop 3: Soft orange
    else: return "#2B8CBE"            # Hop 4+: Steel blue (safe zone)

# ---------------------------------------------------------------------------
# Simulated Transaction Value Generator
# (estimates BTC value based on edge frequency)
# ---------------------------------------------------------------------------
edge_values_btc = {}
node_total_value = Counter()

for (u, v), freq in edge_frequencies.items():
    estimated_btc = freq * random.uniform(0.5, 2.5)
    edge_values_btc[(u, v)] = estimated_btc

    node_total_value[u] += estimated_btc
    node_total_value[v] += estimated_btc

# ---------------------------------------------------------------------------
# 3. Build Graph: Nodes and Edges with Visual Encodings
# ---------------------------------------------------------------------------
for route in jalur_pelarian[:200]:
    for i in range(len(route) - 1):
        u, v = route[i], route[i+1]
        current_hop = i + 1
        layer_color = get_hop_color(current_hop)

        def add_insightful_node(node_id, hop_level_node):
            """Add a node to the network with forensic annotations and styling."""
            if node_id not in added_nodes:
                freq = node_frequencies[node_id]
                calculated_size = min(12 + (freq * 1.5), 45)
                total_money_flow = node_total_value[node_id]

                # Tooltip: simplified without emojis for readability on white background
                hover_node = (
                    f"Address: {node_id}\n"
                    f"Volume: {freq} routes\n"
                    f"Estimation: {total_money_flow:.2f} BTC\n"
                    f"Hop Layer: Hop {hop_level_node}"
                )

                # Visual logic based on role
                if node_id == target_hot_wallet:
                    # Main target: dark red hexagon
                    net.add_node(
                        node_id,
                        label="Main Target\n(Hot Wallet)",
                        title=hover_node,
                        color="#800000",
                        shape="hexagon",
                        size=55
                    )
                elif node_id in cycle_nodes and node_id != target_hot_wallet:
                    # Mixer anomaly: dark purple triangle (indigo)
                    hover_node += "\n[Mixer Cycle Detected]"
                    net.add_node(
                        node_id,
                        label="Cycle Detected",
                        title=hover_node,
                        color="#4B0082",
                        shape="triangle",
                        size=30
                    )
                else:
                    # Regular intermediary: circle with truncated label (first 6 chars)
                    node_color = get_hop_color(hop_level_node)
                    net.add_node(
                        node_id,
                        label=node_id[:6] + "...",
                        title=hover_node,
                        color=node_color,
                        size=calculated_size
                    )

                added_nodes.add(node_id)

        # Add both endpoints of the edge
        add_insightful_node(u, current_hop - 1)
        add_insightful_node(v, current_hop)

        # 3. Edge styling with simulated BTC value
        if (u, v) not in added_edges:
            edge_freq = edge_frequencies[(u, v)]
            btc_value = edge_values_btc[(u, v)]

            thickness = min(1.5 + (edge_freq * 0.8), 15)

            # Hover text for edge
            hover_edge = (
                f"Hop {current_hop-1} -> Hop {current_hop}\n"
                f"Volume: {edge_freq} tx\n"
                f"Estimation: ~{btc_value:.2f} BTC"
            )

            net.add_edge(
                u, v,
                title=hover_edge,
                color=layer_color,
                width=thickness
            )
            added_edges.add((u, v))

# ---------------------------------------------------------------------------
# Physics Optimisation for Force‑Directed Layout
# ---------------------------------------------------------------------------
net.repulsion(
    node_distance=150,
    central_gravity=0.005,
    spring_length=250,
    spring_strength=0.02,
    damping=0.09
)

# ---------------------------------------------------------------------------
# Save and Display
# ---------------------------------------------------------------------------
file_name = "heatmap_plustoken_pyo3.html"   # renamed for consistency
net.write_html(file_name)
print(f"Forensic intelligence graph rendered and saved to {file_name}.")

# Automatically open the visualisation in the system browser.
print("Launching system browser...")
os.system(f"explorer.exe {file_name}")

# end of step 4

=== [STEP 4] INTERACTIVE NETWORK VISUALIZATION (HEATMAP, VOLUME & VALUE) ===
Forensic intelligence graph rendered and saved to heatmap_plustoken_pyo3.html.
Launching system browser...


256

In [ ]:
# Menampilkan semua atribut dan fungsi yang tersedia di dalam objek ForensicEngine
print(dir(engine))

In [ ]:
"""
Step 5 of 5: Benford's Law Anomaly Detection (Real Data) + Methodological Rigor.

This step retrieves the digit frequency distributions (digits 1–9) for the
top‑10 Sybil entities discovered in Step 2 and compares them against the
theoretical Benford's Law distribution. 

[UPDATED] Now includes Mean Absolute Deviation (MAD), p-values (Chi-Square),
and 95% Confidence Intervals (Multinomial Bootstrapping) to ensure strict
methodological rigor for journal publication.
"""

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import math
from scipy.stats import chisquare
import warnings

# Mengabaikan warning jika ada dompet kosong/nol pembagian
warnings.filterwarnings("ignore", category=RuntimeWarning)

print("=== [STEP 5] BENFORD'S LAW ANOMALY DETECTION (RIGOROUS STATS) ===")

# ---------------------------------------------------------------------------
# 1. Retrieve the top‑10 entity addresses
# ---------------------------------------------------------------------------
top_10_wallets = [root for root, size in sorted_clusters[:10]]

# ---------------------------------------------------------------------------
# 2. Fetch digit frequency data from the Rust forensic engine
# ---------------------------------------------------------------------------
benford_data = engine.get_benford_stats(top_10_wallets)

# ---------------------------------------------------------------------------
# 3. Theoretical Benford's Law percentages & proportions
# ---------------------------------------------------------------------------
benford_theory_prop = [math.log10(1 + 1/d) for d in range(1, 10)]
benford_theory_pct = [prop * 100 for prop in benford_theory_prop]
digits = [str(i) for i in range(1, 10)]

# ==========================================
# HELPER FUNCTIONS & STATS CALCULATION
# ==========================================

def to_percentage(counts):
    total = sum(counts)
    return [0] * 9 if total == 0 else [(c / total) * 100 for c in counts]

def calculate_mad(actual_counts, theory_prop):
    total = sum(actual_counts)
    if total == 0: return 0
    actual_prop = [c / total for c in actual_counts]
    mad = sum(abs(a - t) for a, t in zip(actual_prop, theory_prop)) / 9
    return mad

def get_mad_conclusion(mad_score):
    if mad_score <= 0.006:
        return "Normal"
    elif mad_score <= 0.012:
        return "Acceptable"
    elif mad_score <= 0.015:
        return "Marginal"
    else:
        return "<span style='color:#ff4d4d'><b>ANOMALY (Non-Conformity)</b></span>"

def calculate_rigor_stats(counts, theory_prop, iterations=10000):
    """
    Menghitung p-value dan 95% Confidence Interval menggunakan 
    operasi matriks multinomial berkecepatan tinggi.
    """
    total = sum(counts)
    if total == 0: 
        return 1.0, 0.0, 0.0
        
    counts_arr = np.array(counts)
    theory_arr = np.array(theory_prop)
    
    # --- A. p-value (Chi-Square) ---
    expected = theory_arr * total
    expected = np.where(expected == 0, 1e-9, expected) # Mencegah ZeroDivision
    _, p_val = chisquare(f_obs=counts_arr, f_exp=expected)
    
    # --- B. 95% CI untuk MAD (Multinomial Bootstrapping) ---
    actual_prop = counts_arr / total
    # Generate 10,000 sampel secara instan
    bootstrap_samples = np.random.multinomial(total, actual_prop, size=iterations)
    bootstrap_props = bootstrap_samples / total
    
    # Hitung MAD secara vektor (Matrix Operation) untuk kecepatan maksimal
    abs_diff = np.abs(bootstrap_props - theory_arr)
    bootstrap_mads = np.sum(abs_diff, axis=1) / 9
    
    ci_lower = np.percentile(bootstrap_mads, 2.5)
    ci_upper = np.percentile(bootstrap_mads, 97.5)
    
    return p_val, ci_lower, ci_upper

def format_p_value(p_val):
    return "p < 0.01" if p_val < 0.01 else f"p = {p_val:.4f}"

# ==========================================
# DATA PROCESSING & SUBPLOT TITLES
# ==========================================
subplot_titles = []
print("\n--- STATISTICAL VALIDATION RESULTS ---")

# 1. Hitung untuk OVERALL DATASET
if "overall" in benford_data:
    overall_counts = benford_data["overall"]
    overall_pct = to_percentage(overall_counts)
    global_mad = calculate_mad(overall_counts, benford_theory_prop)
    g_pval, g_ci_low, g_ci_high = calculate_rigor_stats(overall_counts, benford_theory_prop)
    global_status = get_mad_conclusion(global_mad)
    
    title_overall = (
        f"<b>OVERALL DATASET</b><br>"
        f"<span style='font-size:10px;'>MAD: {global_mad:.4f} (95% CI: [{g_ci_low:.4f}, {g_ci_high:.4f}])</span><br>"
        f"<span style='font-size:10px;'>{format_p_value(g_pval)} | {global_status}</span>"
    )
    subplot_titles.append(title_overall)
    
    print(f"OVERALL DATASET | MAD: {global_mad:.4f} | CI: [{g_ci_low:.4f}, {g_ci_high:.4f}] | {format_p_value(g_pval)}")

# 2. Hitung untuk TOP-10 TARGETS
for i, wallet in enumerate(top_10_wallets):
    if wallet in benford_data["targets"]:
        counts = benford_data["targets"][wallet]
        wallet_pct = to_percentage(counts)
        mad_score = calculate_mad(counts, benford_theory_prop)
        p_val, ci_low, ci_high = calculate_rigor_stats(counts, benford_theory_prop)
        mad_status = get_mad_conclusion(mad_score)
        
        # Build HTML‑formatted subplot title (ukuran font diperkecil agar muat)
        title = (
            f"Target #{i+1}:<br>"
            f"<span style='font-size:9px;'>{wallet}</span><br>"
            f"<span style='font-size:10px;'>MAD: {mad_score:.4f} (CI: [{ci_low:.4f}, {ci_high:.4f}])</span><br>"
            f"<span style='font-size:10px;'>{format_p_value(p_val)} | {mad_status}</span>"
        )
        
        status_log = "ANOMALY" if mad_score > 0.015 else "Acceptable"
        print(f"Target #{i+1:02d} | MAD: {mad_score:.4f} | CI: [{ci_low:.4f}, {ci_high:.4f}] | {format_p_value(p_val)} | {status_log}")
    else:
        title = f"Target #{i+1}:<br><span style='font-size:9px;'>{wallet}</span><br><span style='font-size:10px;'>No Data</span>"
        
    subplot_titles.append(title)

print("--------------------------------------")

# ==========================================
# PLOTLY DASHBOARD SETUP (2 rows × 6 columns)
# ==========================================
fig = make_subplots(
    rows=2, cols=6,
    subplot_titles=subplot_titles,
    vertical_spacing=0.28,   # Dinaikkan sedikit agar ada ruang untuk teks 5 baris di atas
    horizontal_spacing=0.03
)

# ... [BAGIAN ADD_TRACE SAMA SEPERTI SEBELUMNYA] ...
# Trace: Overall dataset
if "overall" in benford_data:
    fig.add_trace(
        go.Bar(x=digits, y=overall_pct, marker_color='#1f77b4', name="Global Data"),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=digits, y=benford_theory_pct, mode='lines',
                   line=dict(color='black', width=2), showlegend=False),
        row=1, col=1
    )

# Trace: Top-10 wallet clusters
row, col = 1, 2
for i, wallet in enumerate(top_10_wallets):
    if wallet in benford_data["targets"]:
        wallet_pct = to_percentage(benford_data["targets"][wallet])
        
        fig.add_trace(
            go.Bar(x=digits, y=wallet_pct, marker_color='#ff4d4d', showlegend=False),
            row=row, col=col
        )
        fig.add_trace(
            go.Scatter(x=digits, y=benford_theory_pct, mode='lines',
                       line=dict(color='black', width=2), showlegend=False),
            row=row, col=col
        )
    
    col += 1
    if col > 6:
        col = 1
        row += 1


# ---------------------------------------------------------------------------
# Dashboard theme customisation
# ---------------------------------------------------------------------------
fig.update_layout(
    height=650,                      # Disesuaikan menjadi 650 agar pas untuk teks dan grafik
    width=1200,
    title=dict(
        text="<b>BENFORD'S LAW ANALYSIS DASHBOARD (WITH STATISTICAL RIGOR)</b>",
        y=0.98,
        x=0.5,
        xanchor='center',
        yanchor='top'
    ),
    title_font_size=18,
    template="plotly_white",
    showlegend=False,
    margin=dict(t=100, b=20, l=20, r=20) # Margin atas (t) ditambah sedikit untuk judul utama
)

# Fix subplot title font sizes
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=10)

fig.show()

In [ ]:
"""
Step 5 of 5: Benford's Law Anomaly Detection (Real Data) + Methodological Rigor.

This step retrieves the digit frequency distributions (digits 1–9) for the
top‑10 Sybil entities discovered in Step 2 and compares them against the
theoretical Benford's Law distribution. 

[UPDATED] Now includes Mean Absolute Deviation (MAD), p-values (Chi-Square),
and 95% Confidence Intervals (Multinomial Bootstrapping) to ensure strict
methodological rigor for journal publication.
"""

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import math
from scipy.stats import chisquare
import warnings

# Mengabaikan warning jika ada dompet kosong/nol pembagian
warnings.filterwarnings("ignore", category=RuntimeWarning)

print("=== [STEP 5] BENFORD'S LAW ANOMALY DETECTION (RIGOROUS STATS) ===")

# ---------------------------------------------------------------------------
# 1. Retrieve the top‑10 entity addresses
# ---------------------------------------------------------------------------
top_10_wallets = [root for root, size in sorted_clusters[:10]]

# ---------------------------------------------------------------------------
# 2. Fetch digit frequency data from the Rust forensic engine
# ---------------------------------------------------------------------------
benford_data = engine.get_benford_stats(top_10_wallets)

# ---------------------------------------------------------------------------
# 3. Theoretical Benford's Law percentages & proportions
# ---------------------------------------------------------------------------
benford_theory_prop = [math.log10(1 + 1/d) for d in range(1, 10)]
benford_theory_pct = [prop * 100 for prop in benford_theory_prop]
digits = [str(i) for i in range(1, 10)]

# ==========================================
# HELPER FUNCTIONS & STATS CALCULATION
# ==========================================

def to_percentage(counts):
    total = sum(counts)
    return [0] * 9 if total == 0 else [(c / total) * 100 for c in counts]

def calculate_mad(actual_counts, theory_prop):
    total = sum(actual_counts)
    if total == 0: return 0
    actual_prop = [c / total for c in actual_counts]
    mad = sum(abs(a - t) for a, t in zip(actual_prop, theory_prop)) / 9
    return mad

def get_mad_conclusion(mad_score):
    if mad_score <= 0.006:
        return "Normal"
    elif mad_score <= 0.012:
        return "Acceptable"
    elif mad_score <= 0.015:
        return "Marginal"
    else:
        return "<span style='color:#ff4d4d'><b>ANOMALY (Non-Conformity)</b></span>"

def calculate_rigor_stats(counts, theory_prop, iterations=10000):
    """
    Menghitung p-value dan 95% Confidence Interval menggunakan 
    operasi matriks multinomial berkecepatan tinggi.
    """
    total = sum(counts)
    if total == 0: 
        return 1.0, 0.0, 0.0
        
    counts_arr = np.array(counts)
    theory_arr = np.array(theory_prop)
    
    # --- A. p-value (Chi-Square) ---
    expected = theory_arr * total
    expected = np.where(expected == 0, 1e-9, expected) # Mencegah ZeroDivision
    _, p_val = chisquare(f_obs=counts_arr, f_exp=expected)
    
    # --- B. 95% CI untuk MAD (Multinomial Bootstrapping) ---
    actual_prop = counts_arr / total
    # Generate 10,000 sampel secara instan
    bootstrap_samples = np.random.multinomial(total, actual_prop, size=iterations)
    bootstrap_props = bootstrap_samples / total
    
    # Hitung MAD secara vektor (Matrix Operation) untuk kecepatan maksimal
    abs_diff = np.abs(bootstrap_props - theory_arr)
    bootstrap_mads = np.sum(abs_diff, axis=1) / 9
    
    ci_lower = np.percentile(bootstrap_mads, 2.5)
    ci_upper = np.percentile(bootstrap_mads, 97.5)
    
    return p_val, ci_lower, ci_upper

def format_p_value(p_val):
    return "p < 0.01" if p_val < 0.01 else f"p = {p_val:.4f}"

# ==========================================
# DATA PROCESSING & SUBPLOT TITLES
# ==========================================
subplot_titles = []
print("\n--- STATISTICAL VALIDATION RESULTS ---")

# 1. Hitung untuk OVERALL DATASET
if "overall" in benford_data:
    overall_counts = benford_data["overall"]
    overall_pct = to_percentage(overall_counts)
    global_mad = calculate_mad(overall_counts, benford_theory_prop)
    g_pval, g_ci_low, g_ci_high = calculate_rigor_stats(overall_counts, benford_theory_prop)
    global_status = get_mad_conclusion(global_mad)
    
    title_overall = (
        f"<b>OVERALL DATASET</b><br>"
        f"<span style='font-size:10px;'>MAD: {global_mad:.4f} (95% CI: [{g_ci_low:.4f}, {g_ci_high:.4f}])</span><br>"
        f"<span style='font-size:10px;'>{format_p_value(g_pval)} | {global_status}</span>"
    )
    subplot_titles.append(title_overall)
    
    print(f"OVERALL DATASET | MAD: {global_mad:.4f} | CI: [{g_ci_low:.4f}, {g_ci_high:.4f}] | {format_p_value(g_pval)}")

# 2. Hitung untuk TOP-10 TARGETS
for i, wallet in enumerate(top_10_wallets):
    if wallet in benford_data["targets"]:
        counts = benford_data["targets"][wallet]
        wallet_pct = to_percentage(counts)
        mad_score = calculate_mad(counts, benford_theory_prop)
        p_val, ci_low, ci_high = calculate_rigor_stats(counts, benford_theory_prop)
        mad_status = get_mad_conclusion(mad_score)
        
        # Build HTML‑formatted subplot title (ukuran font diperkecil agar muat)
        title = (
            f"Target #{i+1}:<br>"
            f"<span style='font-size:9px;'>{wallet}</span><br>"
            f"<span style='font-size:10px;'>MAD: {mad_score:.4f} (CI: [{ci_low:.4f}, {ci_high:.4f}])</span><br>"
            f"<span style='font-size:10px;'>{format_p_value(p_val)} | {mad_status}</span>"
        )
        
        status_log = "ANOMALY" if mad_score > 0.015 else "Acceptable"
        print(f"Target #{i+1:02d} | MAD: {mad_score:.4f} | CI: [{ci_low:.4f}, {ci_high:.4f}] | {format_p_value(p_val)} | {status_log}")
    else:
        title = f"Target #{i+1}:<br><span style='font-size:9px;'>{wallet}</span><br><span style='font-size:10px;'>No Data</span>"
        
    subplot_titles.append(title)

print("--------------------------------------")

# ==========================================
# PLOTLY DASHBOARD SETUP (3 rows × 4 columns)
# ==========================================
fig = make_subplots(
    rows=3, cols=4,
    subplot_titles=subplot_titles,
    vertical_spacing=0.15,   # Sedikit direnggangkan agar teks CI/P-value tidak terpotong
    horizontal_spacing=0.05
)

# Trace: Overall dataset
if "overall" in benford_data:
    fig.add_trace(
        go.Bar(x=digits, y=overall_pct, marker_color='#1f77b4', name="Global Data"),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=digits, y=benford_theory_pct, mode='lines',
                   line=dict(color='black', width=2), showlegend=False),
        row=1, col=1
    )

# Trace: Top‑10 wallet clusters
row, col = 1, 2
for i, wallet in enumerate(top_10_wallets):
    if wallet in benford_data["targets"]:
        wallet_pct = to_percentage(benford_data["targets"][wallet])
        
        fig.add_trace(
            go.Bar(x=digits, y=wallet_pct, marker_color='#ff4d4d', showlegend=False),
            row=row, col=col
        )
        fig.add_trace(
            go.Scatter(x=digits, y=benford_theory_pct, mode='lines',
                       line=dict(color='black', width=2), showlegend=False),
            row=row, col=col
        )
    
    col += 1
    if col > 4:
        col = 1
        row += 1

# ---------------------------------------------------------------------------
# Dashboard theme customisation
# ---------------------------------------------------------------------------
fig.update_layout(
    height=700,                      # Ditinggikan sedikit untuk menampung teks statistik
    width=1350,
    title=dict(
        text="<b>BENFORD'S LAW ANALYSIS DASHBOARD (WITH STATISTICAL RIGOR)</b>",
        y=0.98,
        x=0.5,
        xanchor='center',
        yanchor='top'
    ),
    title_font_size=20,
    template="plotly_white",
    showlegend=False,
    margin=dict(t=120, b=40, l=40, r=40)
)

# Fix subplot title font sizes
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=12)

fig.show()
print("\n[✓] Rigorous Dashboard generated successfully! ")